In [12]:
from pathlib import Path
import numpy as np
import pandas as pd

OUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/output")
parquet_path = OUT_DIR / "ait_ads_wazuh.parquet"

df_wazuh = pd.read_parquet(parquet_path)
print("Loaded df_wazuh:", df_wazuh.shape)
df_wazuh.head(3)


Loaded df_wazuh: (2600263, 14)


,file,timestamp,location,agent_id,agent_name,agent_ip,hostname,program,full_log,rule_id,rule_description,rule_groups,rule_level,rule_groups_str
0,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
1,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
2,fox_wazuh.json,2022-01-15 02:32:37+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"


In [13]:
import pandas as pd

def map_priority(level):
    if pd.isna(level):
        return None
    level = float(level)
    if level <= 3:
        return "low"
    elif level <= 6:
        return "medium"
    elif level <= 10:
        return "high"
    else:
        return "critical"

df = df_wazuh.copy()
df["y_priority"] = df["rule_level"].apply(map_priority)

df_ml = df.dropna(subset=["full_log", "y_priority"]).copy()
print("After dropna:", df_ml.shape)
display(df_ml["y_priority"].value_counts())


After dropna: (2293628, 15)


y_priority
medium      1873973
low          286087
high         131901
critical       1667
Name: count, dtype: int64

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

N_SAMPLE = 200_000  # Вариант A: увеличиваем выборку
df_s = df_ml.sample(n=min(N_SAMPLE, len(df_ml)), random_state=42).copy()

X_text = df_s["full_log"].astype("string").fillna("").tolist()
y_text = df_s["y_priority"].astype("string").tolist()

le = LabelEncoder()
y = le.fit_transform(y_text)
print("Classes:", list(le.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", len(X_train), "Test:", len(X_test))
print("Train class counts:", np.bincount(y_train))
print("Test class counts :", np.bincount(y_test))


Classes: [np.str_('critical'), np.str_('high'), np.str_('low'), np.str_('medium')]
Train: 160000 Test: 40000
Train class counts: [   129   9351  20008 130512]
Test class counts : [   32  2338  5002 32628]


In [15]:
import re
from collections import Counter

def tokenize(s: str):
    s = s.lower()
    # токены для логов (ip, пути, порты, идентификаторы)
    return re.findall(r"[a-z0-9\.\_\:\-/]+", s)

MAX_VOCAB = 50_000
MAX_LEN = 128
PAD, UNK = 0, 1

counter = Counter()
for t in X_train:
    counter.update(tokenize(t))

stoi = {"<pad>": PAD, "<unk>": UNK}
for i, (tok, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    stoi[tok] = i

def encode_text(s: str):
    ids = [stoi.get(tok, UNK) for tok in tokenize(s)]
    ids = ids[:MAX_LEN]
    if len(ids) < MAX_LEN:
        ids += [PAD] * (MAX_LEN - len(ids))
    return ids

# int32 экономит память, ускоряет CPU
X_train_ids = np.array([encode_text(t) for t in X_train], dtype=np.int32)
X_test_ids  = np.array([encode_text(t) for t in X_test], dtype=np.int32)

print("Vocab size:", len(stoi))
print("X_train_ids:", X_train_ids.shape, "X_test_ids:", X_test_ids.shape)


Vocab size: 50000
X_train_ids: (160000, 128) X_test_ids: (40000, 128)


In [16]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)

# Вариант A: ограничиваем веса, чтобы не было "medium -> critical" массово
cw = np.clip(cw, 0.5, 10.0)

class_weights = {int(c): float(w) for c, w in zip(classes, cw)}
print("class_weights (balanced, capped):", class_weights)


class_weights (balanced, capped): {0: 10.0, 1: 4.277617367126511, 2: 1.9992003198720512, 3: 0.5}


In [17]:
import torch
from torch.utils.data import Dataset, DataLoader

device = "cpu"

class LogDataset(Dataset):
    def __init__(self, X_ids, y):
        self.X = torch.tensor(X_ids, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): 
        return len(self.y)
    def __getitem__(self, i): 
        return self.X[i], self.y[i]

train_ds = LogDataset(X_train_ids, y_train)
test_ds  = LogDataset(X_test_ids, y_test)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=256, shuffle=False)

xb, yb = next(iter(train_dl))
print("Batch shape:", xb.shape, yb.shape)


Batch shape: torch.Size([128, 128]) torch.Size([128])


In [18]:
import torch.nn as nn

class BiLSTM_FIXED(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        e = self.emb(x)                        # [B,T,E]
        out, _ = self.lstm(e)                  # [B,T,2H]
        mask = (x != 0).float().unsqueeze(-1)  # [B,T,1]
        out = out * mask
        denom = mask.sum(dim=1)                # [B,1]
        pooled = out.sum(dim=1) / denom.clamp(min=1.0)  # [B,2H]
        pooled = self.dropout(pooled)
        return self.fc(pooled)

vocab_size = len(stoi)
num_classes = len(le.classes_)

model = BiLSTM_FIXED(
    vocab_size=vocab_size,
    emb_dim=128,
    hidden_dim=128,
    num_classes=num_classes,
    pad_idx=PAD
).to(device)

weights_tensor = torch.tensor(
    [class_weights.get(i, 1.0) for i in range(num_classes)],
    dtype=torch.float32
)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)

print("Model ready. Num classes:", num_classes)


Model ready. Num classes: 4


In [19]:
from sklearn.metrics import f1_score
import numpy as np
import torch

def run_epoch(train=True, log_every=100):
    model.train(train)
    dl = train_dl if train else test_dl

    total_loss = 0.0
    all_pred, all_true = [], []

    for step, (xb, yb) in enumerate(dl, start=1):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # стабилизация LSTM
            optimizer.step()

        total_loss += loss.item() * xb.size(0)
        all_pred.append(logits.argmax(dim=1).detach().cpu().numpy())
        all_true.append(yb.detach().cpu().numpy())

        if step % log_every == 0:
            print(("train" if train else "eval"),
                  f"step {step}/{len(dl)} loss={loss.item():.4f}")

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    return total_loss / len(dl.dataset), y_true, y_pred

EPOCHS = 8
best_f1 = -1.0
patience = 2
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, _, _ = run_epoch(train=True, log_every=200)
    te_loss, y_true, y_pred = run_epoch(train=False, log_every=200)

    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"\nEpoch {epoch}/{EPOCHS} | train_loss={tr_loss:.4f} | test_loss={te_loss:.4f} | macro_f1={macro_f1:.4f}")

    if macro_f1 > best_f1 + 1e-4:
        best_f1 = macro_f1
        bad_epochs = 0
        torch.save(model.state_dict(), "best_lstm.pt")
        print("✅ saved best_lstm.pt")
    else:
        bad_epochs += 1
        print(f"⚠ no improvement ({bad_epochs}/{patience})")
        if bad_epochs >= patience:
            print("🛑 Early stopping.")
            break

# загрузим лучшую модель
model.load_state_dict(torch.load("best_lstm.pt", map_location=device))
print("Loaded best model with macro_f1:", best_f1)


train step 200/1250 loss=0.4288
train step 400/1250 loss=0.5318
train step 600/1250 loss=0.5589
train step 800/1250 loss=0.4747
train step 1000/1250 loss=0.5118
train step 1200/1250 loss=0.4896

Epoch 1/8 | train_loss=0.5086 | test_loss=0.5002 | macro_f1=0.4912
✅ saved best_lstm.pt
train step 200/1250 loss=0.5276
train step 400/1250 loss=0.5635
train step 600/1250 loss=0.5198
train step 800/1250 loss=0.4822
train step 1000/1250 loss=0.3801
train step 1200/1250 loss=0.4831

Epoch 2/8 | train_loss=0.4861 | test_loss=0.5074 | macro_f1=0.4937
✅ saved best_lstm.pt
train step 200/1250 loss=0.2943
train step 400/1250 loss=0.3202
train step 600/1250 loss=0.5091
train step 800/1250 loss=0.4386
train step 1000/1250 loss=0.4175
train step 1200/1250 loss=0.3558

Epoch 3/8 | train_loss=0.4259 | test_loss=0.5627 | macro_f1=0.5009
✅ saved best_lstm.pt
train step 200/1250 loss=0.3267
train step 400/1250 loss=0.5445
train step 600/1250 loss=0.3834
train step 800/1250 loss=0.3486
train step 1000/1250 lo

In [20]:
from sklearn.metrics import classification_report, confusion_matrix

# финальная оценка на тесте
_, y_true, y_pred = run_epoch(train=False, log_every=200)

print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


              precision    recall  f1-score   support

    critical       0.00      0.00      0.00        32
        high       0.07      0.08      0.08      2338
         low       1.00      1.00      1.00      5002
      medium       0.93      0.92      0.93     32628

    accuracy                           0.88     40000
   macro avg       0.50      0.50      0.50     40000
weighted avg       0.89      0.88      0.89     40000

Confusion matrix:
 [[    0     0     0    32]
 [    0   196     0  2142]
 [    0     0  5000     2]
 [    1  2563     0 30064]]
